# fGPU Fractional GPU 공유 실험 — 추론 노트북 (A/B 공통)

**사용법**: 맨 위 config 셀의 `SESSION_LABEL`, `SCENARIO`, `RATIO` 만 바꾸고  
`Kernel → Restart & Run All` 하면 됩니다.

- **SESSION_LABEL**: 이 세션이 `"A"` 인지 `"B"` 인지 구분자.  
- **SCENARIO**: `"solo"` / `"seq"` / `"concurrent"` 중 하나.  
- **RATIO**: 세션 생성 시 지정한 `FGPU_RATIO` 값을 기록용으로 입력 (측정에는 영향 없음).

결과는 `/workspace/session_result_<SCENARIO>.csv` 와  
`/workspace/session_meta_<SCENARIO>.json` 에 저장됩니다.  
분석 노트북(`fgpu_analysis.ipynb`)이 두 세션의 파일을 합쳐 비교합니다.

> **한계 사전 고지**  
> - SM 격리 없음: 두 컨테이너가 동일 SM 을 자유 경쟁합니다.  
> - `PYTORCH_NO_CUDA_MEMORY_CACHING=1` 기본값: KV cache 매 스텝 cudaMalloc/Free → generate() 5~30x 저하.  
>   solo/concurrent 모두 동일 조건이므로 **speedup 상대 비교는 유효**합니다.  
> - pynvml per-process GPU 사용량은 컨테이너 PID namespace 불일치로 신뢰 불가 →  
>   `torch.cuda.max_memory_allocated()` 를 주 메모리 지표로 사용합니다.  
> - 동시 시작은 "수동 동시 Shift+Enter" 이므로 수 초 skew 가 발생할 수 있습니다.  
>   makespan 계산에 사용하는 `t_start_epoch` 는 배리어 통과 직후 찍어 skew 를 최소화합니다.


In [ ]:
# ══════════════════════════════════════════════════════
# CONFIG — 실험 전 여기만 수정
# ══════════════════════════════════════════════════════

# 세션 식별자: "A" 또는 "B"
SESSION_LABEL = "A"

# 시나리오: "solo" | "seq" | "concurrent"
SCENARIO = "solo"

# 이 세션에 적용된 FGPU_RATIO (기록용, 실제 quota 는 세션 spawn 시 주입됨)
RATIO = 0.5

# 사용할 HuggingFace 모델 ID
# 권장: Qwen2-0.5B-Instruct (bf16 ~1 GB)
# 대안 주석:
#   "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  — ~2.2 GB bf16, 더 빡빡
#   "microsoft/phi-2"                      — ~5.5 GB, 12 GB 예산 빡빡
#   "distilgpt2"                           — ~300 MB, 드라마 없음 (참고용)
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"

# 추론 설정
N_ITERS = 20          # 측정 반복 횟수
MAX_NEW_TOKENS = 128  # 생성할 최대 토큰 수
WARMUP = 3            # 측정 전 워밍업 횟수 (결과에 포함 안 됨)

# 고정 프롬프트 (solo/concurrent 동일 조건 비교를 위해 고정)
PROMPT = (
    "Explain the concept of fractional GPU resource sharing in cloud computing. "
    "Describe how multiple workloads can share a single GPU efficiently."
)

# pynvml 폴링 간격 (초)
POLL_INTERVAL = 0.05   # 50ms

# concurrent 시나리오: 배리어 대기 타임아웃 (초)
BARRIER_TIMEOUT = 120


In [ ]:
# ══════════════════════════════════════════════════════
# 설치 — 멱등 (이미 깔려 있으면 무해)
# ══════════════════════════════════════════════════════
import sys

# transformers: Qwen2 모델 로드
# accelerate: device_map="auto" 지원, 대형 모델 분산 로드
# nvidia-ml-py: pynvml GPU util 폴링
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "accelerate", "nvidia-ml-py"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("[fgpu-infer] pip install 실패:", result.stderr[:500])
else:
    print("[fgpu-infer] 패키지 설치/확인 완료")

# HF 캐시 경로: /workspace/hf_cache (bind-mount 된 영속 경로)
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.makedirs("/workspace/hf_cache", exist_ok=True)
print(f"[fgpu-infer] HF_HOME = {os.environ['HF_HOME']}")


In [ ]:
# ══════════════════════════════════════════════════════
# 환경 출력 — hook / quota 컨텍스트 확인
# ══════════════════════════════════════════════════════
import os
import torch

print("=" * 60)
print(f"SESSION_LABEL            : {SESSION_LABEL}")
print(f"SCENARIO                 : {SCENARIO}")
print(f"RATIO (기록용)           : {RATIO}")
print("-" * 60)

# CUDA 가용 여부
cuda_ok = torch.cuda.is_available()
print(f"torch.cuda.is_available(): {cuda_ok}")
if cuda_ok:
    print(f"GPU 디바이스             : {torch.cuda.get_device_name(0)}")
    total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"GPU 총 메모리 (MiB)      : {total_mib:.0f}")
    quota_mib = total_mib * RATIO
    print(f"예상 quota @ RATIO={RATIO} : {quota_mib:.0f} MiB")
else:
    print("[경고] CUDA 없음 — '--gpus all' 누락 또는 hook 미주입 의심")

print("-" * 60)

# fGPU hook 관련 환경변수
fgpu_vars = [
    "PYTORCH_NO_CUDA_MEMORY_CACHING",
    "LD_PRELOAD",
    "FGPU_RATIO",
    "FGPU_QUOTA_BYTES",
    "FGPU_THROTTLE_ENABLE",
    "FGPU_COMPUTE_RATIO",
    "FGPU_LAUNCH_LOG_EVERY",
]
for var in fgpu_vars:
    val = os.environ.get(var, "<unset>")
    print(f"{var:<38}: {val}")

print("-" * 60)
# PYTORCH_NO_CUDA_MEMORY_CACHING 주의 메모
caching_val = os.environ.get("PYTORCH_NO_CUDA_MEMORY_CACHING", "0")
if caching_val == "1":
    print("[주의] PYTORCH_NO_CUDA_MEMORY_CACHING=1 활성 상태입니다.")
    print("       KV cache 매 스텝이 cudaMalloc/Free 로 직접 들어가")
    print("       generate() 속도가 실배포 대비 5~30x 느릴 수 있습니다.")
    print("       solo/concurrent 모두 동일 조건이므로 speedup 상대값은 유효합니다.")
else:
    print("[주의] PYTORCH_NO_CUDA_MEMORY_CACHING 미설정 (caching ON).")
    print("       hook quota 가 caching allocator 에 의해 masking 될 수 있습니다.")
    print("       실배포 조건에 가깝지만 hook 효과 관측이 어렵습니다.")

print("=" * 60)


In [ ]:
# ══════════════════════════════════════════════════════
# pynvml GPU 폴링 유틸 — 배경 스레드
# ══════════════════════════════════════════════════════
# 주의: pynvml.nvmlDeviceGetComputeRunningProcesses() 의 PID 는
#       호스트 PID namespace 기준이라 컨테이너 내부 PID 와 다릅니다.
#       따라서 per-process 분해는 불가 — GPU 전체 util/mem 을
#       두 컨테이너 합산 보조 proxy 로만 사용합니다.
#       주 메모리 지표는 torch.cuda.max_memory_allocated() 입니다.

import threading
import time as _time

try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    PYNVML_OK = True
    print("[fgpu-infer] pynvml 초기화 완료")
except Exception as e:
    PYNVML_OK = False
    print(f"[fgpu-infer] pynvml 비활성: {e}")
    print("             GPU util 시계열은 기록되지 않습니다 (torch.cuda 메모리 지표는 정상).")


class GpuPoller:
    """generate() 실행 중 GPU util% 와 used mem(MiB) 을 백그라운드 폴링."""

    def __init__(self, interval: float = 0.05):
        self.interval = interval
        self._stop = threading.Event()
        # 각 샘플: (timestamp_epoch, util_pct, used_mem_mib)
        self.samples: list[tuple[float, int, float]] = []
        self._thread = threading.Thread(target=self._run, daemon=True)

    def start(self):
        self._stop.clear()
        self.samples.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self, timeout: float = 2.0):
        self._stop.set()
        self._thread.join(timeout=timeout)

    def _run(self):
        while not self._stop.is_set():
            ts = _time.time()
            if PYNVML_OK:
                try:
                    util = pynvml.nvmlDeviceGetUtilizationRates(_nvml_handle).gpu
                    mem_info = pynvml.nvmlDeviceGetMemoryInfo(_nvml_handle)
                    used_mib = mem_info.used / 1024**2
                    self.samples.append((ts, util, used_mib))
                except Exception:
                    pass
            _time.sleep(self.interval)

    @property
    def avg_util(self) -> float:
        utils = [s[1] for s in self.samples]
        return sum(utils) / len(utils) if utils else 0.0

    @property
    def peak_used_mib(self) -> float:
        mems = [s[2] for s in self.samples]
        return max(mems) if mems else 0.0

    def as_dict_list(self) -> list[dict]:
        return [
            {"ts": s[0], "util_pct": s[1], "used_mem_mib": s[2]}
            for s in self.samples
        ]


_poller = GpuPoller(interval=POLL_INTERVAL)
print("[fgpu-infer] GpuPoller 준비 완료")


In [ ]:
# ══════════════════════════════════════════════════════
# 모델 로드
# ══════════════════════════════════════════════════════
# HF 캐시: /workspace/hf_cache (세션 workspace bind-mount 내 영속 경로)
# 첫 실행 시 모델 다운로드 (~1 GB). 이후 캐시 사용.
#
# OOM 처리:
#   - torch.cuda.OutOfMemoryError  → hook DENY 또는 물리 OOM
#   - 기타 예외                    → 일반 오류
#
# H4 케이스: 두 LLM 동시 로드 시 CUDA context(~700 MiB × 2) +
#   가중치(~1 GB × 2) = ~3.4 GB. quota=0.5×12282=6141 MiB 이내라
#   정상 로드 가능. KV cache 를 키워야 OOM 유발 (H4 시연은 별도 셀).

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_OOM = False   # 로드 실패 플래그
model = None
tokenizer = None

print(f"[fgpu-infer] 모델 로드 시작: {MODEL_ID}")
print(f"[fgpu-infer] HF_HOME = {os.environ.get('HF_HOME', '<unset>')}")

torch.cuda.reset_peak_memory_stats()

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,   # bf16: ~1 GB for Qwen2-0.5B
        device_map="cuda:0",          # 단일 GPU
    )
    model.eval()

    load_peak_mib = torch.cuda.max_memory_allocated() / 1024**2
    print(f"[fgpu-infer] 모델 로드 완료")
    print(f"[fgpu-infer] 로드 후 peak_mem_allocated = {load_peak_mib:.1f} MiB")

    # quota 대비 실사용률 (admission 직교성)
    if torch.cuda.is_available():
        total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
        quota_mib = total_mib * RATIO
        usage_ratio = load_peak_mib / quota_mib if quota_mib > 0 else 0
        print(f"[fgpu-infer] 실사용 / quota = {load_peak_mib:.0f} / {quota_mib:.0f} MiB"
              f" ({usage_ratio*100:.1f}%)")
        print(f"[fgpu-infer] 주의: admission ratio_used=1.0 이어도 실사용은 {usage_ratio*100:.1f}%")
        print(f"             — 이것이 admission 직교성(예약 ≠ 실사용)의 핵심 포인트입니다.")

except torch.cuda.OutOfMemoryError as e:
    MODEL_OOM = True
    print(f"[fgpu-infer] OOM — 모델 로드 실패 (hook DENY 또는 물리 한계)")
    print(f"             {e}")
    print(f"             H4 시나리오: admission 통과 + 물리 OOM 관측됨")
except Exception as e:
    MODEL_OOM = True
    print(f"[fgpu-infer] 모델 로드 오류: {type(e).__name__}: {e}")

if MODEL_OOM:
    print("[fgpu-infer] 로드 실패로 이후 셀은 의미 없음. 결과에 OOM 기록.")


In [ ]:
# ══════════════════════════════════════════════════════
# concurrent 시나리오 배리어 (방법 B — 파일 플래그)
# ══════════════════════════════════════════════════════
# 경로 1(코드 0 변경): 공유 마운트(/barrier) 없이 /workspace 내
# 플래그 파일로 동기화합니다.
#
# 동작:
#   1) 이 세션이 준비됐다는 플래그 파일을 /workspace 에 씁니다.
#   2) concurrent 시나리오에서는 "동시 Shift+Enter" 로 두 노트북을 같이 실행.
#      두 세션의 /workspace 가 서로 다른 디렉토리여서 상대 플래그를 읽기 어렵습니다.
#      따라서 경로 1에서는 간단히 모델 로드 완료 시각을 기록하고
#      실험 메타에 포함해 사후 정렬합니다.
#   3) SCENARIO == "concurrent" 이면 "지금 동시에 run 중입니다" 출력.
#      makespan 계산은 분석 노트북이 t_start_epoch/t_end_epoch 로 정렬.
#
# 배리어 방법 A (공유 마운트, M2 구현 예정) 사용 시에는 이 셀을 교체하세요:
#   BARRIER_DIR = pathlib.Path("/barrier")
#   (BARRIER_DIR / f"ready_{SESSION_LABEL}.flag").touch()
#   while len(list(BARRIER_DIR.glob("ready_*.flag"))) < 2: time.sleep(0.05)
#   t_barrier_passed = time.time()

import time as _time_mod

_model_ready_epoch = _time_mod.time()

if SCENARIO == "concurrent":
    print(f"[fgpu-infer] ★ concurrent 모드 ★")
    print(f"[fgpu-infer] 모델 로드 완료 시각: {_model_ready_epoch:.3f}")
    print(f"[fgpu-infer] 상대 세션(B 또는 A)의 노트북도 이 시점에 실행 중이어야 합니다.")
    print(f"[fgpu-infer] 두 세션을 거의 동시에 'Run All' 했다면 skew < 5s 로 허용합니다.")
    print(f"[fgpu-infer] makespan_conc = max(t_end_A, t_end_B) - min(t_start_A, t_start_B)")
elif SCENARIO == "solo":
    print(f"[fgpu-infer] solo 모드: 상대 세션 없음. 기준선 측정.")
elif SCENARIO == "seq":
    print(f"[fgpu-infer] seq 모드: 이전 세션이 완료·삭제된 후 이 세션 실행.")
else:
    print(f"[fgpu-infer] 알 수 없는 SCENARIO='{SCENARIO}'. solo 처럼 진행합니다.")

print(f"[fgpu-infer] 모델 준비 완료, 측정 셀로 진행합니다.")


In [ ]:
# ══════════════════════════════════════════════════════
# 추론 측정
# ══════════════════════════════════════════════════════
import time
import torch

# 모델 로드 실패면 측정 건너뜀
if MODEL_OOM or model is None:
    print("[fgpu-infer] 모델 OOM/미로드 — 측정 건너뜀")
    _iter_results = []
    _meas_start_epoch = time.time()
    _meas_end_epoch = time.time()
    _peak_alloc_mib = 0.0
    _peak_reserved_mib = 0.0
    _gpu_util_avg = 0.0
    _gpu_util_samples_n = 0
    _poller_data = []
else:
    # 토크나이저 입력 준비
    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda:0")
    prompt_tokens = inputs["input_ids"].shape[-1]
    print(f"[fgpu-infer] 프롬프트 토큰 수: {prompt_tokens}")
    print(f"[fgpu-infer] MAX_NEW_TOKENS:   {MAX_NEW_TOKENS}")
    print(f"[fgpu-infer] WARMUP:           {WARMUP}")
    print(f"[fgpu-infer] N_ITERS:          {N_ITERS}")
    print()

    # ── 워밍업 ──────────────────────────────────────────
    print("[fgpu-infer] 워밍업 시작...")
    with torch.no_grad():
        for w in range(WARMUP):
            _ = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,   # greedy: 재현성 보장
                pad_token_id=tokenizer.eos_token_id,
            )
    torch.cuda.synchronize()
    print(f"[fgpu-infer] 워밍업 {WARMUP}회 완료")
    print()

    # ── 본 측정 ─────────────────────────────────────────
    torch.cuda.reset_peak_memory_stats()

    # pynvml 폴링 시작
    _poller.start()

    _meas_start_epoch = time.time()
    _iter_results = []   # (latency_s, gen_tokens, tokens_per_s, iter_start, iter_end)

    print("[fgpu-infer] 본 측정 시작...")
    with torch.no_grad():
        for i in range(N_ITERS):
            _t0 = time.perf_counter()
            _t0_epoch = time.time()

            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            torch.cuda.synchronize()

            _t1 = time.perf_counter()
            _t1_epoch = time.time()

            latency_s = _t1 - _t0
            gen_tokens = out.shape[-1] - inputs["input_ids"].shape[-1]
            tps = gen_tokens / latency_s if latency_s > 0 else 0.0

            _iter_results.append((_t0_epoch, _t1_epoch, latency_s, gen_tokens, tps))

            if (i + 1) % 5 == 0 or i == 0:
                print(f"  iter {i+1:3d}/{N_ITERS}: {latency_s:.2f}s, "
                      f"{gen_tokens} tokens, {tps:.1f} tok/s")

    _meas_end_epoch = time.time()
    torch.cuda.synchronize()
    _poller.stop()

    _peak_alloc_mib    = torch.cuda.max_memory_allocated()  / 1024**2
    _peak_reserved_mib = torch.cuda.max_memory_reserved()   / 1024**2
    _gpu_util_avg      = _poller.avg_util
    _gpu_util_samples_n = len(_poller.samples)
    _poller_data       = _poller.as_dict_list()

    print()
    print(f"[fgpu-infer] 측정 완료: {N_ITERS}회")
    print(f"[fgpu-infer] 경과 시간: {_meas_end_epoch - _meas_start_epoch:.2f}s")


In [ ]:
# ══════════════════════════════════════════════════════
# 통계 계산
# ══════════════════════════════════════════════════════
import statistics

if not _iter_results:
    # OOM 등 측정 실패
    _stats = {
        "oom": True,
        "n_iters": 0,
        "makespan_s": 0.0,
        "mean_latency_s": 0.0,
        "p50_latency_s": 0.0,
        "p95_latency_s": 0.0,
        "mean_tokens_per_s": 0.0,
        "total_gen_tokens": 0,
        "t_start_epoch": _meas_start_epoch,
        "t_end_epoch": _meas_end_epoch,
    }
else:
    latencies = [r[2] for r in _iter_results]
    tps_list  = [r[4] for r in _iter_results]
    gen_toks  = [r[3] for r in _iter_results]

    sorted_lat = sorted(latencies)
    n = len(sorted_lat)

    def _pct(data, p):
        """선형 보간 백분위수."""
        if not data: return 0.0
        idx = (len(data) - 1) * p / 100
        lo, hi = int(idx), min(int(idx) + 1, len(data) - 1)
        return data[lo] + (data[hi] - data[lo]) * (idx - lo)

    _stats = {
        "oom": False,
        "n_iters": n,
        "makespan_s": _meas_end_epoch - _meas_start_epoch,
        "mean_latency_s":  statistics.mean(latencies),
        "p50_latency_s":   _pct(sorted_lat, 50),
        "p95_latency_s":   _pct(sorted_lat, 95),
        "mean_tokens_per_s": statistics.mean(tps_list) if tps_list else 0.0,
        "total_gen_tokens": sum(gen_toks),
        "t_start_epoch": _meas_start_epoch,
        "t_end_epoch":   _meas_end_epoch,
    }

print("통계 계산 완료")


In [ ]:
# ══════════════════════════════════════════════════════
# 결과 저장
# ══════════════════════════════════════════════════════
# 파일명에 SCENARIO 포함 → 같은 세션에서 여러 시나리오 실행 가능.
# 분석 노트북은 glob("*/session_result_*.csv") 로 수집.

import csv
import json
import os
import time as _time_save

RESULT_CSV  = f"/workspace/session_result_{SCENARIO}.csv"
META_JSON   = f"/workspace/session_meta_{SCENARIO}.json"

# ── CSV 저장 ──────────────────────────────────────────
csv_header = [
    "label", "scenario", "ratio", "model",
    "iter", "iter_start_epoch", "iter_end_epoch",
    "latency_s", "gen_tokens", "tokens_per_s",
]

with open(RESULT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_header)
    if _iter_results:
        for idx, (t0e, t1e, lat, gtok, tps) in enumerate(_iter_results):
            writer.writerow([
                SESSION_LABEL,
                SCENARIO,
                RATIO,
                MODEL_ID,
                idx + 1,
                f"{t0e:.6f}",
                f"{t1e:.6f}",
                f"{lat:.6f}",
                gtok,
                f"{tps:.4f}",
            ])
    else:
        # OOM 케이스: 한 행으로 실패 기록
        writer.writerow([
            SESSION_LABEL, SCENARIO, RATIO, MODEL_ID,
            0, f"{_meas_start_epoch:.6f}", f"{_meas_end_epoch:.6f}",
            0.0, 0, 0.0,
        ])

print(f"[fgpu-infer] CSV 저장: {RESULT_CSV}")

# ── 메타 JSON 저장 ────────────────────────────────────
# env 스냅샷 (hook 컨텍스트 보존)
env_snap = {
    k: os.environ.get(k, None)
    for k in [
        "PYTORCH_NO_CUDA_MEMORY_CACHING",
        "LD_PRELOAD",
        "FGPU_RATIO",
        "FGPU_QUOTA_BYTES",
        "FGPU_THROTTLE_ENABLE",
        "FGPU_COMPUTE_RATIO",
        "FGPU_LAUNCH_LOG_EVERY",
    ]
}

meta = {
    "label": SESSION_LABEL,
    "scenario": SCENARIO,
    "ratio": RATIO,
    "model": MODEL_ID,
    "oom": MODEL_OOM or _stats.get("oom", False),
    "n_iters": _stats["n_iters"],
    "makespan_s": _stats["makespan_s"],
    "t_start_epoch": _stats["t_start_epoch"],
    "t_end_epoch":   _stats["t_end_epoch"],
    "mean_latency_s":    _stats["mean_latency_s"],
    "p50_latency_s":     _stats["p50_latency_s"],
    "p95_latency_s":     _stats["p95_latency_s"],
    "mean_tokens_per_s": _stats["mean_tokens_per_s"],
    "total_gen_tokens":  _stats["total_gen_tokens"],
    "peak_mem_alloc_mib":    _peak_alloc_mib,
    "peak_mem_reserved_mib": _peak_reserved_mib,
    "gpu_util_avg_pct":   _gpu_util_avg,
    "gpu_util_samples_n": _gpu_util_samples_n,
    # pynvml 시계열 (보조 지표, 파일 크기 고려해 최대 500 샘플만)
    "gpu_util_timeline": _poller_data[:500],
    "env_snapshot": env_snap,
    "saved_at": _time_save.strftime("%Y-%m-%dT%H:%M:%SZ", _time_save.gmtime()),
}

with open(META_JSON, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[fgpu-infer] 메타 저장: {META_JSON}")


In [ ]:
# ══════════════════════════════════════════════════════
# 요약 출력 — 분석 노트북 없이도 한눈에 확인
# ══════════════════════════════════════════════════════
import time as _t_sum

print("=" * 60)
print(f"  fGPU 추론 실험 요약")
print(f"  세션: {SESSION_LABEL}  시나리오: {SCENARIO}  RATIO: {RATIO}")
print("=" * 60)

if MODEL_OOM or not _iter_results:
    print("  상태 : OOM / 측정 실패")
    print(f"  파일 : {RESULT_CSV}")
    print(f"         {META_JSON}")
else:
    print(f"  모델            : {MODEL_ID}")
    print(f"  반복 횟수       : {_stats['n_iters']} 회")
    print(f"  총 경과 시간    : {_stats['makespan_s']:.2f} s")
    print(f"  mean latency    : {_stats['mean_latency_s']:.3f} s")
    print(f"  p50  latency    : {_stats['p50_latency_s']:.3f} s")
    print(f"  p95  latency    : {_stats['p95_latency_s']:.3f} s")
    print(f"  mean tokens/s   : {_stats['mean_tokens_per_s']:.2f}")
    print(f"  총 생성 토큰    : {_stats['total_gen_tokens']}")
    print(f"  peak mem alloc  : {_peak_alloc_mib:.1f} MiB")
    print(f"  peak mem rsv    : {_peak_reserved_mib:.1f} MiB")
    print(f"  GPU util avg    : {_gpu_util_avg:.1f}% "
          f"({_gpu_util_samples_n} samples, 보조 지표)")
    print()

    if torch.cuda.is_available():
        total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
        quota_mib = total_mib * RATIO
        print(f"  [admission 직교성]")
        print(f"  예약 ratio       : {RATIO} ({quota_mib:.0f} MiB)")
        print(f"  실사용 peak      : {_peak_alloc_mib:.1f} MiB"
              f" ({_peak_alloc_mib/quota_mib*100:.1f}% of quota)")
        print(f"  → 예약 {RATIO*100:.0f}% 이지만 실사용은 quota의"
              f" {_peak_alloc_mib/quota_mib*100:.1f}%")

    print()
    if _stats["mean_tokens_per_s"] > 0:
        caching_on = os.environ.get("PYTORCH_NO_CUDA_MEMORY_CACHING", "0") != "1"
        if not caching_on:
            print("  [해석 주의]")
            print("  PYTORCH_NO_CUDA_MEMORY_CACHING=1 — 절대 tokens/sec는 실배포보다 느림.")
            print("  solo ↔ concurrent speedup 상대 비교는 유효합니다.")

print()
print(f"  결과 파일: {RESULT_CSV}")
print(f"  메타 파일: {META_JSON}")
print("=" * 60)
print("  다음 단계: fgpu_analysis.ipynb 로 A/B 시나리오 비교 분석")
print("=" * 60)
